# Notebook: S1 documentation from GenBank

This notebook builds an **S1 documentation Excel** from exported **GenBank (`.gb/.gbk`)** plasmids.  
It extracts *Promoter / RBS / Gene* (preferring **features**, otherwise the **file name**), infers the **vector/antibiotic marker** (Amp/Kan → DVA/DVK), resolves the **donor organism**, and writes a clean table to the current working directory.

---

## What it does

- Scans a folder for `*.gb` / `*.gbk` files (e.g. `reports/Assembly/`).
- Parses **Promoter**, **RBS**, **Gene** from feature qualifiers; falls back to simple filename heuristics (`J23…`, `B00…`, `PanD/ADC`, …).
- Infers **backbone / selection marker** from feature text (Ampicillin → DVA, Kanamycin → DVK).
- Resolves **donor organism** robustly (see *Donor resolution* below).
- Writes **`S1_Dokumentation.xlsx`** with standard S1 columns to the **current notebook directory**.

---

## Donor resolution (robust)

1. **From file**: `record.annotations["organism"]` or `/source` qualifiers.  
   Placeholders like `.`, `-`, `unknown`, *synthetic construct* are treated as **missing**.
2. **Curated mapping (optional)**: If you provide a CSV mapping (e.g. plasmid ID → donor), it is used next.
3. **NCBI fallback (optional)**: If still unknown and an accession is present in the filename, query NCBI (with caching) and **review** results.

If all steps fail, the code uses your **`default_donor`** parameter.

---

## Inputs

- **`folder_path`**: directory containing GenBank files (e.g. `../reports/Assembly`).
- **Optional glob** filter to include a subset (e.g. only TU1: `*TU1*.gb*`).

---

## Outputs

- **Excel**: `S1_Dokumentation.xlsx` (written to the **current working directory**).

**Columns written:**

---

## Quick start
```python
# Example call (assuming the function is defined in this notebook)
out = build_s1_documentation(
    folder_path="../reports/Assembly",  # folder with .gb/.gbk
    include_glob=None,                  # e.g. "*TU1*.gb*" to filter
    prefer_features=True,               # use feature annotations first
    recipients="E. coli BL21, E. coli Top10, E. coli DH5alpha",
    donor="C. glutamicum",
    rg_donor=1,
    rg_recipient=1,
    rg_construct=1,
    output_name="S1_Dokumentation.xlsx",
)
out
```

## Requirements
- `biopython`, `pandas`, `openpyxl` /`docx` (for Excel/Word export) and `cairosvg`

---

## Notes
- If feature annotations are missing or inconsistent, the notebook falls back to the **filename heuristic**.
- You can tweak the heuristics in the helper functions if your naming scheme differs.


### 1. Imports

In [1]:
import pandas as pd
from pathlib import Path
from typing import Iterable, Optional, Sequence, Mapping
from pathlib import Path
import os

# S1 Excel table
from assembly_designer.plasmidio import build_plasmid_catalog_docx
from assembly_designer.plasmidio import build_s1_documentation

### 2. Helper functions 

In [2]:
# helpers: write donor mapping CSV for testing

def write_donor_mapping_csv(
    out_path: str | Path,
    entries: Sequence[Mapping[str, str]],
) -> Path:
    """
    Write a donor-mapping CSV used by `resolve_donor(...)`.

    Parameters
    ----------
    out_path : str | Path
        Where to write the CSV (e.g. "donor_mapping.csv").
    entries : sequence of mappings
        Each item must have at least:
          - 'stem'  : file stem (without .gb/.gbk)
          - 'donor' : organism name
        Optional keys:
          - 'accession' : GenBank accession (if you want to match by accession)
          - 'gene'      : gene symbol/name (fallback key)

    Returns
    -------
    Path
        Absolute path to the written CSV.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Preserve only supported columns; missing ones are fine
    cols = ["stem", "donor", "accession", "gene"]
    df = pd.DataFrame(entries)
    df = df[[c for c in cols if c in df.columns]]

    df.to_csv(out_path, index=False)
    return out_path



### 3. Design donor_mapping.csv

In [3]:
# --- Example usage -----------------------------------------------------------
rows = [
    {
        "stem": "J23107_AB_BCD12_BC_DVA_ecPanD_B0015_DF_DVK_AF",
        "donor": "Escherichia coli",
        "gene": "panD",                # optional
        # "accession": "XYZ12345",     # optional
    },
    {
        "stem": "TU1_J23106_TU2_B0032_TU3_GFP_B0015",
        "donor": "Corynebacterium glutamicum",
    },
]

csv_path = write_donor_mapping_csv("donor_mapping.csv", rows)
print("Wrote:", csv_path.resolve())

Wrote: C:\Users\tim\repos\assembly_designer\examples\04 S1 Documentation\01 Example\donor_mapping.csv


### 4. build_plasmid_catalog as .docx

In [4]:
gb_folder = r"..\..\02 insilico Plasmid Construction\01 Golden Gate Cloning_All_Combinations\01 Example\reports\Assembly"

In [5]:
# Simple example usage:

build_plasmid_catalog_docx(gb_folder, render_maps=False)



WindowsPath('c:/Users/tim/repos/assembly_designer/examples/04 S1 Documentation/01 Example/Plasmid_Database.docx')

In [6]:
# advanced example usage:
out_docx = build_plasmid_catalog_docx(
    gb_folder,
    output_name="Plasmid_Catalog.docx",
    render_maps=True,          # set False if you don’t want any plots
    figure_width=5.0,          # map figure size during rendering
    skip_feature_types=("source","homology"),
)
out_docx

WindowsPath('c:/Users/tim/repos/assembly_designer/examples/04 S1 Documentation/01 Example/Plasmid_Catalog.docx')

### 5. Build Excel table for S1 doc

In [7]:
# Example usage
out = build_s1_documentation(
    r"..\..\02 insilico Plasmid Construction\01 Golden Gate Cloning_All_Combinations\01 Example\reports\Assembly",
    donor_mapping_csv=Path("donor_mapping.csv"),  # optional curated mapping
    enable_ncbi_fallback=True,                     # opt-in
    ncbi_email="t.stoltmann@fz-juelich.de",       # required by NCBI
    ncbi_api_key=os.getenv("NCBI_API_KEY"),       # optional
    ncbi_cache=Path.home()/".cache/assembly_designer/ncbi_cache.json",
)
print("Wrote:", out)


Wrote: c:\Users\tim\repos\assembly_designer\examples\04 S1 Documentation\01 Example\S1_Dokumentation.xlsx
